In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, recall_score, precision_score,
 f1_score, roc_auc_score,classification_report,
    confusion_matrix, RocCurveDisplay)



In [2]:
def load_processed_data():
    """Carregar dados após feature engineering"""

    PROCESSED_DATA_PATH_X = ("../data/processed/X_features.csv")
    PROCESSED_DATA_PATH_y = ("../data/processed/y_target.csv")
    
    X = pd.read_csv(PROCESSED_DATA_PATH_X)
    y = pd.read_csv(PROCESSED_DATA_PATH_y)
    
    # Se y for DataFrame com uma coluna, converter para Series
    if isinstance(y, pd.DataFrame) and y.shape[1] == 1:
        y = y.iloc[:, 0]
    
    print(f"✅ Dados carregados: X {X.shape}, y {y.shape}")
    return X, y

In [3]:
load_processed_data()

✅ Dados carregados: X (302, 28), y (302,)


(     age  sex  cp  trestbps  chol  fbs  restecg  thalach  exang  oldpeak  ...  \
 0     52    1   0       125   212    0        1      168      0      1.0  ...   
 1     53    1   0       140   203    1        0      155      1      3.1  ...   
 2     70    1   0       145   174    0        1      125      1      2.6  ...   
 3     61    1   0       148   203    0        1      161      0      0.0  ...   
 4     62    0   0       138   294    1        1      106      0      1.9  ...   
 ..   ...  ...  ..       ...   ...  ...      ...      ...    ...      ...  ...   
 297   68    0   2       120   211    0        0      115      0      1.5  ...   
 298   44    0   2       108   141    0        1      175      0      0.6  ...   
 299   52    1   0       128   255    0        1      161      1      0.0  ...   
 300   59    1   3       160   273    0        0      125      0      0.0  ...   
 301   54    1   0       120   188    0        1      113      0      1.4  ...   
 
      age_grou

In [4]:
PROCESSED_DATA_PATH_X = ("../data/processed/X_features.csv")
PROCESSED_DATA_PATH_y = ("../data/processed/y_target.csv")
    
X = pd.read_csv(PROCESSED_DATA_PATH_X)
y = pd.read_csv(PROCESSED_DATA_PATH_y)

In [5]:
smote = SMOTE(random_state=42)
X_balanced, y_balanced = smote.fit_resample(X, y)

# Verificação de balanceamento
#ANTES
print("\nDistribuição de classes antes:")
print(y.value_counts())
#DEPOIS 
# Criar novo dataframe balanceado
data_balanced = pd.DataFrame(X_balanced, columns=X.columns)
data_balanced['Target'] = y_balanced

# Verificar balanceamento
print("Distribuição após SMOTE:\n", data_balanced['Target'].value_counts())


Distribuição de classes antes:
target
1         164
0         138
Name: count, dtype: int64
Distribuição após SMOTE:
 Target
0    164
1    164
Name: count, dtype: int64


In [6]:
print(f"✅ Dados balanceados: X {X_balanced.shape}, y {y_balanced.shape}")  
data_balanced.head()

✅ Dados balanceados: X (328, 28), y (328, 1)


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,...,bp_category_Hypertension_Crisis,bp_category_Hypertension_Stage1,bp_category_Hypertension_Stage2,bp_category_Normal,chol_category_Borderline,chol_category_High,hr_performance_Normal,hr_performance_Good,hr_performance_Excellent,Target
0,52,1,0,125,212,0,1,168,0,1.0,...,False,False,False,False,True,False,False,True,False,0
1,53,1,0,140,203,1,0,155,1,3.1,...,False,False,True,False,True,False,False,True,False,0
2,70,1,0,145,174,0,1,125,1,2.6,...,False,False,True,False,False,False,True,False,False,0
3,61,1,0,148,203,0,1,161,0,0.0,...,False,False,True,False,True,False,False,False,True,0
4,62,0,0,138,294,1,1,106,0,1.9,...,False,True,False,False,False,True,True,False,False,0


In [7]:
X = data_balanced.drop('Target', axis=1)  # Features
y = data_balanced['Target']              # TargetA

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [8]:
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [9]:
y_pred = model.predict(X_test)


In [10]:
# 4. Avaliar o Modelo
print("Acurácia:", accuracy_score(y_test, y_pred))
print("Precisão:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1-score:", f1_score(y_test, y_pred))

# Matriz de Confusão
print("\nMatriz de Confusão:")
print(confusion_matrix(y_test, y_pred))

# Relatório de Classificação
print("\nRelatório de Classificação:")
print(classification_report(y_test, y_pred))

Acurácia: 0.8080808080808081
Precisão: 0.7169811320754716
Recall: 0.9047619047619048
F1-score: 0.8

Matriz de Confusão:
[[42 15]
 [ 4 38]]

Relatório de Classificação:
              precision    recall  f1-score   support

           0       0.91      0.74      0.82        57
           1       0.72      0.90      0.80        42

    accuracy                           0.81        99
   macro avg       0.82      0.82      0.81        99
weighted avg       0.83      0.81      0.81        99



In [11]:
param_grid = {
    'n_estimators': [50, 100, 200],  # Número de árvores
    'max_depth': [None, 10, 20],     # Profundidade máxima das árvores
    'min_samples_split': [2, 5, 10], # Mínimo de amostras para dividir um nó
    'min_samples_leaf': [1, 2, 4]    # Mínimo de amostras em uma folha
}

grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=5, scoring='f1', n_jobs=-1)
grid_search.fit(X_train, y_train)

# Melhores hiperparâmetros
print("\nMelhores Hiperparâmetros:", grid_search.best_params_)


Melhores Hiperparâmetros: {'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 100}


In [12]:
best_model = grid_search.best_estimator_
y_pred_best = best_model.predict(X_test)

print("\nAvaliação do Modelo Otimizado:")
print("Acurácia:", accuracy_score(y_test, y_pred_best))
print("Precisão:", precision_score(y_test, y_pred_best))
print("Recall:", recall_score(y_test, y_pred_best))
print("F1-score:", f1_score(y_test, y_pred_best))

# Relatório de Classificação do Modelo Otimizado
print("\nRelatório de Classificação (Modelo Otimizado):")
print(classification_report(y_test, y_pred_best))


Avaliação do Modelo Otimizado:
Acurácia: 0.7777777777777778
Precisão: 0.6923076923076923
Recall: 0.8571428571428571
F1-score: 0.7659574468085106

Relatório de Classificação (Modelo Otimizado):
              precision    recall  f1-score   support

           0       0.87      0.72      0.79        57
           1       0.69      0.86      0.77        42

    accuracy                           0.78        99
   macro avg       0.78      0.79      0.78        99
weighted avg       0.80      0.78      0.78        99



In [15]:
# 6. ROC-AUC e Curva ROC
y_pred_proba = best_model.predict_proba(X_test)[:, 1]  # Probabilidades da classe positiva (1)
roc_auc = roc_auc_score(y_test, y_pred_proba)
print("\nROC-AUC Score:", roc_auc)

RocCurveDisplay.from_estimator(best_model, X_test, y_test)
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Classificador Aleatório (AUC = 0.5)')  # Linha diagonal
plt.title('Curva ROC')
plt.legend(loc='lower right')
plt.show()


ROC-AUC Score: 0.9114452798663324


C:\Users\gyova\AppData\Local\Temp\ipykernel_39332\1629979057.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [16]:
# Validação cruzada com F1-score
scores = cross_val_score(best_model, X, y, cv=5, scoring='f1')
print("Validação Cruzada (F1-score):", scores.mean())
print("Desvio Padrão:", scores.std())

Validação Cruzada (F1-score): 0.8392975073785103
Desvio Padrão: 0.023133753532116087


In [17]:
# Extrair importância das features
importances = best_model.feature_importances_
feature_importance_df = pd.DataFrame({'Feature': X.columns, 'Importance': importances})

# Ordenar por importância
feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)
print(feature_importance_df)

# Plotar importância das features
plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=feature_importance_df)
plt.title('Importância das Features')
plt.show()

                            Feature  Importance
2                                cp    0.133638
7                           thalach    0.098698
9                           oldpeak    0.097838
13                hr_percentage_max    0.090079
11                               ca    0.080534
12                             thal    0.079884
14                       risk_score    0.056756
10                            slope    0.046340
0                               age    0.044952
8                             exang    0.043710
4                              chol    0.040061
3                          trestbps    0.034256
1                               sex    0.028259
25            hr_performance_Normal    0.026317
6                           restecg    0.014920
20  bp_category_Hypertension_Stage1    0.009612
26              hr_performance_Good    0.009570
23         chol_category_Borderline    0.009525
22               bp_category_Normal    0.007656
24               chol_category_High    0

C:\Users\gyova\AppData\Local\Temp\ipykernel_39332\208157601.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [23]:
exemplo = {
    'age': 52,                           # Idade em anos (ex: 52)
    'sex': 0,                          # Sexo biológico (M/F)
    'cp': 2,                             # Tipo de dor no peito (0-3)
    'trestbps': 145,                     # Pressão arterial em repouso (mmHg)
    'chol': 230,                         # Colesterol sérico (mg/dl)
    'fbs': 1,                            # Glicemia em jejum > 120 mg/dl (0=False, 1=True)
    'restecg': 1,                        # Resultado eletrocardiográfico em repouso (0-2)
    'thalach': 150,                      # Frequência cardíaca máxima alcançada
    'exang': 0,                          # Angina induzida por exercício (0=Não, 1=Sim)
    'oldpeak': 1.2,                      # Depressão do segmento ST induzida por exercício
    'slope': 2,                          # Inclinação do segmento ST no pico (1-3)
    'ca': 1,                             # Número de vasos principais coloridos (0-3)
    'thal': 3,                           # Talassemia (1-3)
    'hr_percentage_max': 0.85,           # % da frequência cardíaca máxima alcançada (0-1)
    'risk_score': 12.5,                  # Pontuação de risco calculada
    'age_group_40-50': False,            # Pertence ao grupo 40-50 anos (True/False)
    'age_group_50-60': True,             # Pertence ao grupo 50-60 anos (True/False)
    'age_group_60-70': False,            # Pertence ao grupo 60-70 anos (True/False)
    'age_group_70_mais': False,          # Pertence ao grupo 70+ anos (True/False)
    'bp_category_Hypertension_Crisis': False,    # Crise hipertensiva (True/False)
    'bp_category_Hypertension_Stage1': True,     # Hipertensão Estágio 1 (True/False)
    'bp_category_Hypertension_Stage2': False,    # Hipertensão Estágio 2 (True/False)
    'bp_category_Normal': False,                 # Pressão normal (True/False)
    'chol_category_Borderline': True,    # Colesterol limítrofe (True/False)
    'chol_category_High': False,         # Colesterol alto (True/False)
    'hr_performance_Normal': False,      # Desempenho cardíaco normal (True/False)
    'hr_performance_Good': True,         # Bom desempenho cardíaco (True/False)
    'hr_performance_Excellent': False    # Desempenho cardíaco excelente (True/False)
}

# Converter o exemplo em um array numpy
import numpy as np
exemplo_array = np.array([list(exemplo.values())])

# Fazer a predição
predicao = best_model.predict(exemplo_array)

# Exibir o resultado
if predicao[0] == 1:
    print("Predição: risco")
else:
    print("Predição: baixo risco")

Predição: risco


c:\Users\gyova\OneDrive\Documentos\Projeto 1 - Monitoramento de Doenças\venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


In [26]:
exemplo_baixo_risco = {
    'age': 38,                           # Idade jovem - fator protetor
    'sex': 1,                          # Sexo feminino - menor risco pré-menopausa
    'cp': 0,                             # Sem dor torácica
    'trestbps': 112,                     # Pressão arterial ótima
    'chol': 160,                         # Colesterol desejável
    'fbs': 0,                            # Glicemia normal em jejum
    'restecg': 0,                        # ECG normal em repouso
    'thalach': 175,                      # Frequência cardíaca máxima excelente
    'exang': 0,                          # Sem angina induzida por exercício
    'oldpeak': 0.5,                      # Depressão ST mínima
    'slope': 2,                          # Inclinação ascendente - sinal positivo
    'ca': 0,                             # Nenhum vaso afetado
    'thal': 1,                           # Fluxo normal
    'hr_percentage_max': 0.92,           # Alta porcentagem da FC máxima alcançada
    'risk_score': 2.3,                   # Pontuação de risco muito baixa
    'age_group_40-50': False,
    'age_group_50-60': False,
    'age_group_60-70': False,
    'age_group_70_mais': False,
    'bp_category_Hypertension_Crisis': False,
    'bp_category_Hypertension_Stage1': False,
    'bp_category_Hypertension_Stage2': False,
    'bp_category_Normal': True,          # Pressão normal
    'chol_category_Borderline': False,
    'chol_category_High': False,         # Colesterol normal
    'hr_performance_Normal': False,
    'hr_performance_Good': False,
    'hr_performance_Excellent': True     # Desempenho cardíaco excelente
}

# Converter o exemplo em um array numpy
import numpy as np
exemplo_array = np.array([list(exemplo_baixo_risco.values())])

# Fazer a predição
predicao = best_model.predict(exemplo_array)

# Exibir o resultado
if predicao[0] == 1:
    print("Predição: risco")
else:
    print("Predição: baixo risco")

Predição: baixo risco


c:\Users\gyova\OneDrive\Documentos\Projeto 1 - Monitoramento de Doenças\venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
